# Milestone 5 — W8A8 End-to-End

FP16 baseline → W8A8 calibration → generate 비교

In [ ]:
!pip install datasets -q

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
model.eval()
print(f"device: {DEVICE}, dtype: {model.dtype}")

## 1. FP16 Baseline Generate

`initialize()` 전에 실행 — 모델이 in-place로 수정되기 때문

In [ ]:
PROMPT = "The key to artificial intelligence is"
MAX_NEW_TOKENS = 50

inputs = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    fp_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

fp_text = tokenizer.decode(fp_ids[0], skip_special_tokens=True)
print("[FP16]")
print(fp_text)

## 2. Calibration Dataloader 구성

wikitext-2, 32샘플, 256토큰

In [ ]:
from datasets import load_dataset

NUM_SAMPLES = 32
SEQ_LEN = 256

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
text = "\n\n".join(t for t in dataset["text"] if len(t.strip()) > 0)

token_ids = tokenizer(text, return_tensors="pt")["input_ids"][0]

calib_data = []
for i in range(NUM_SAMPLES):
    chunk = token_ids[i * SEQ_LEN : (i + 1) * SEQ_LEN].unsqueeze(0).to(DEVICE)
    calib_data.append({"input_ids": chunk})

print(f"calibration samples: {len(calib_data)}, seq_len: {calib_data[0]['input_ids'].shape[1]}")

## 3. W8A8 적용 — initialize → calibrate → finalize

In [ ]:
from mini_compressor.modifier import QuantizationModifier
from mini_compressor.schemes import W8A8

modifier = QuantizationModifier(model, W8A8, ignore=["lm_head"])

modifier.initialize()
print("initialize() 완료")

modifier.calibrate(calib_data)
print("calibrate() 완료")

modifier.finalize()
print("finalize() 완료")

In [ ]:
# 대표 레이어 scale 확인
for name, mod in model.named_modules():
    from mini_compressor.fake_quant_linear import FakeQuantLinear
    if isinstance(mod, FakeQuantLinear):
        print(f"{name}: weight_scale={mod.weight_scale.shape}, input_scale={mod.input_scale}")
        break

## 4. W8A8 Generate

In [ ]:
with torch.no_grad():
    w8a8_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

w8a8_text = tokenizer.decode(w8a8_ids[0], skip_special_tokens=True)
print("[W8A8]")
print(w8a8_text)

## 5. FP16 vs W8A8 비교

In [ ]:
print("=" * 60)
print("[FP16]")
print(fp_text)
print()
print("[W8A8]")
print(w8a8_text)
print("=" * 60)
print(f"\n출력 일치 여부: {fp_text == w8a8_text}")